In [1]:
!pip install gymnasium tabulate torch

In [4]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from tabulate import tabulate

In [2]:
graphs = [
    {
        "name": "G1",
        "adj_matrix": [
            [0, 1, 0, 0],
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0]
        ],
        "start": 0,
        "end": 3
    },
    {
        "name": "G2",
        "adj_matrix": [
            # 0 1 2 3 4 5 6 7 8
            [0, 1, 1, 0, 0, 0, 0, 0, 0],
            [1, 0, 0, 1, 1, 0, 0, 0, 0],
            [1, 0, 0, 0, 1, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0, 1, 1, 0],
            [0, 1, 1, 0, 0, 1, 0, 0, 0],
            [0, 0, 0, 0, 1, 0, 0, 0, 1],
            [0, 0, 0, 1, 0, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 1, 0, 0, 0]
        ],
        "start": 0,
        "end": 8
    }
]

In [5]:
class GraphEnv(gym.Env):

    def __init__(self, adj_matrix, start_node, end_node):
        super().__init__()
        self.adj_matrix = np.array(adj_matrix)
        self.num_nodes = len(adj_matrix)
        self.start_node = start_node
        self.end_node = end_node

        self.current_node = start_node
        self.action_space = spaces.Discrete(self.num_nodes)
        self.observation_space = spaces.Discrete(self.num_nodes)

        self.max_steps = self.num_nodes * 2
        self.step_count = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_node = self.start_node
        self.step_count = 0
        return self.current_node, {}

    def step(self, action):
        self.step_count += 1
        terminated = False
        truncated = False

        if self.adj_matrix[self.current_node][action] == 1:
            self.current_node = action
            reward = -1
        else:
            reward = -10

        if self.current_node == self.end_node:
            terminated = True
            reward = 0

        if self.step_count >= self.max_steps:
            truncated = True

        return self.current_node, reward, terminated, truncated, {}

In [6]:
results = []

for g in graphs:
    env = GraphEnv(g["adj_matrix"], g["start"], g["end"])
    q_table = np.zeros((env.num_nodes, env.num_nodes))

    learning_rate = 0.1
    discount = 0.95
    epochs = 1000
    epsilon = 1.0
    epsilon_decay = 0.99

    start_time = time.time()

    for _ in range(epochs):
        state, _ = env.reset()
        terminated = truncated = False

        while not (terminated or truncated):
            action = env.action_space.sample() if random.random() < epsilon else np.argmax(q_table[state])
            new_state, reward, terminated, truncated, _ = env.step(action)
            q_table[state, action] += learning_rate * (reward + discount * np.max(q_table[new_state]) - q_table[state, action])
            state = new_state

        epsilon = max(0.01, epsilon * epsilon_decay)

    elapsed = time.time() - start_time

    path = [g["start"]]
    state = g["start"]
    while state != g["end"]:
        action = int(np.argmax(q_table[state]))
        path.append(action)
        state = action
        if len(path) > env.num_nodes:
            break

    results.append([g["name"], "RL", str(path), len(path)-1, f"{elapsed:.2f}s"])

In [7]:
class QNetwork(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n)
        )
    def forward(self, x): return self.layers(x)

def one_hot(n, size):
    v = np.zeros(size)
    v[n] = 1
    return v

for g in graphs:
    env = GraphEnv(g["adj_matrix"], g["start"], g["end"])
    n = env.num_nodes

    q = QNetwork(n).to("cpu")
    target = QNetwork(n).to("cpu")
    target.load_state_dict(q.state_dict())
    opt = optim.Adam(q.parameters(), lr=0.001)

    memory = deque(maxlen=5000)
    epsilon = 1.0

    start = time.time()

    for _ in range(500):
        state, _ = env.reset()
        s = one_hot(state, n)
        done = False

        while not done:
            if random.random() < epsilon:
                action = random.randrange(n)
            else:
                with torch.no_grad(): action = q(torch.tensor(s).float()).argmax().item()

            next_state, reward, term, trunc, _ = env.step(action)
            s2 = one_hot(next_state, n)
            memory.append((s, action, reward, s2, term or trunc))
            s = s2
            done = term or trunc

            if len(memory) >= 32:
                batch = random.sample(memory, 32)
                S, A, R, S2, D = zip(*batch)
                S = torch.tensor(S).float()
                A = torch.tensor(A).long().unsqueeze(1)
                R = torch.tensor(R).float().unsqueeze(1)
                S2 = torch.tensor(S2).float()
                D = torch.tensor(D).float().unsqueeze(1)

                qv = q(S).gather(1, A)
                with torch.no_grad():
                    q2 = target(S2).max(1)[0].unsqueeze(1)
                tq = R + 0.95 * q2 * (1 - D)

                loss = nn.MSELoss()(qv, tq)
                opt.zero_grad(); loss.backward(); opt.step()

                for tp, lp in zip(target.parameters(), q.parameters()):
                    tp.data.copy_(tp.data * 0.99 + lp.data * 0.01)

        epsilon = max(0.01, epsilon * 0.995)

    elapsed = time.time() - start

    path = [g["start"]]
    s = one_hot(g["start"], n)
    st = g["start"]
    while st != g["end"]:
        with torch.no_grad(): a = q(torch.tensor(s).float()).argmax().item()
        path.append(a)
        st = a
        s = one_hot(a, n)
        if len(path) > n: break

    results.append([g["name"], "DRL", str(path), len(path)-1, f"{elapsed:.2f}s"])

/tmp/ipython-input-632701144.py:50: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  S = torch.tensor(S).float()


In [8]:
print(tabulate(results, headers=["Graph","Method","Path","Path Length","Time"], tablefmt="grid"))

+---------+----------+-----------------+---------------+--------+
| Graph   | Method   | Path            |   Path Length | Time   |
+=========+==========+=================+===============+========+
| G1      | RL       | [0, 1, 2, 3]    |             3 | 0.03s  |
+---------+----------+-----------------+---------------+--------+
| G2      | RL       | [0, 1, 4, 5, 8] |             4 | 0.04s  |
+---------+----------+-----------------+---------------+--------+
| G1      | DRL      | [0, 1, 2, 3]    |             3 | 4.57s  |
+---------+----------+-----------------+---------------+--------+
| G2      | DRL      | [0, 1, 4, 5, 8] |             4 | 8.36s  |
+---------+----------+-----------------+---------------+--------+
